# 01 · Data cleaning: Feeding America Map the Meal Gap (MMG)

**Goal:** get Minnesota county food-insecurity estimates for **2022, 2023, 2024** (the years Q1 asks for).

**Source files** (in `Data/ORIGINAL_MMG/`, never edited):

| Data year | File | Notes |
|---|---|---|
| 2022, 2023 | `MMG2025_2019-2023_Data_To_Share_v2.xlsx` | v2 = **corrected** 2023 county estimates (updated 2026-07-28, see `MMG_2025_County_Revisions_Detail.pdf`) |
| 2024 | `MMG2026_2024_Data_To_Share.xlsx` | |

**Outputs:**
1. `Data/processed/mmg_mn_county_2022_2024.csv`: cleaned + combined, **use this one for analysis**
2. `Data/local/MMG_State_MN_2022-2024_raw.csv`: statewide headline numbers (local only, not pushed)

In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MMG_DIR = ROOT / "Data" / "ORIGINAL_MMG"
OUT_DIR = ROOT / "Data" / "processed"
LOCAL_DIR = ROOT / "Data" / "local"
OUT_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DIR.mkdir(parents=True, exist_ok=True)

FILE_2025 = "MMG2025_2019-2023_Data_To_Share_v2.xlsx"  # has 2019-2023
FILE_2026 = "MMG2026_2024_Data_To_Share.xlsx"          # has 2024

## Step 1: Look at the original files

Each Excel file has several tabs. We use the **County** tab (one row = one US county in one year).
The **State** tab (one row = one whole state in one year) is used only for headline numbers in Step 5.

In [2]:
for f in [FILE_2025, FILE_2026]:
    print(f, "->", pd.ExcelFile(MMG_DIR / f).sheet_names)

county_2025 = pd.read_excel(MMG_DIR / FILE_2025, sheet_name="County", dtype={"FIPS": str})
county_2026 = pd.read_excel(MMG_DIR / FILE_2026, sheet_name="County", dtype={"FIPS": str})
print("\n2025 file County tab:", county_2025.shape, "| years:", sorted(county_2025["Year"].unique()))
print("2026 file County tab:", county_2026.shape, "| years:", sorted(county_2026["Year"].unique()))
county_2025.head(3)

MMG2025_2019-2023_Data_To_Share_v2.xlsx -> ['Read Me', 'County', 'Congressional District', 'State', 'Metro Area (50+ Only)', 'National (50+ Only)']
MMG2026_2024_Data_To_Share.xlsx -> ['Read Me', 'County', 'Congressional District', 'State', 'Metro Area (50+ Only)', 'National (50+ Only)']



2025 file County tab: (15872, 25) | years: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]
2026 file County tab: (3144, 25) | years: [np.int64(2024)]


,FIPS,State,"County, State",Year,Overall Food Insecurity Rate,Ratio (1 in X) Overall,# of Food Insecure Persons Overall,Food Insecurity Rate among Black Persons (all ethnicities),Food Insecurity Rate among Hispanic Persons (any race),"Food Insecurity Rate among White, non-Hispanic Persons",...,% food insecure children in HH w/ HH incomes below 185 FPL,% food insecure children in HH w/ HH incomes above 185 FPL,Cost Per Meal,Weighted weekly $ needed by FI,Weighted Annual Food Budget Shortfall,Rural-Urban Continuum Code (2013),Rural-Urban Continuum Code (2023),Census Region,Census Division,FNS Region
0,01001,AL,"Autauga County, Alabama",2019,0.157,6.0,8670.0,0.26,NaN,0.12,...,0.69,0.31,3.00,16.88,4439000.0,2.0,NaN,South,East South Central,Southeast
1,01001,AL,"Autauga County, Alabama",2020,0.145,7.0,8070.0,0.25,NaN,0.09,...,0.71,0.29,3.22,17.09,4184000.0,2.0,NaN,South,East South Central,Southeast
2,01001,AL,"Autauga County, Alabama",2021,0.133,8.0,7770.0,0.23,NaN,0.09,...,0.68,0.32,3.60,20.99,4947000.0,2.0,NaN,South,East South Central,Southeast


## Step 2: Filter to Minnesota and our years (no other changes)

The County tab covers **every US county**, so we keep rows where `State == "MN"`, then keep only the years we need.
Every Minnesota year should have exactly **87 counties**.

In [3]:
mn_2022_2023 = county_2025[(county_2025["State"] == "MN") & (county_2025["Year"].isin([2022, 2023]))]
mn_2024 = county_2026[(county_2026["State"] == "MN") & (county_2026["Year"] == 2024)]

print("2022-2023 rows:", len(mn_2022_2023), mn_2022_2023["Year"].value_counts().sort_index().to_dict())
print("2024 rows:     ", len(mn_2024))
assert len(mn_2022_2023) == 87 * 2 and len(mn_2024) == 87

mn_2024.head(3)

2022-2023 rows: 174 {2022: 87, 2023: 87}
2024 rows:      87


,FIPS,State,"County, State",Year,Overall Food Insecurity Rate,# of Food Insecure Persons Overall,Food Insecurity Rate among Black Persons (all ethnicities),Food Insecurity Rate among Hispanic Persons (any race),"Food Insecurity Rate among White, non-Hispanic Persons",SNAP Threshold,...,% food insecure children in HH w/ HH incomes above 185 FPL,Imputed Market Basket Data,Cost Per Meal,Weighted weekly $ needed by FI,Weighted Annual Food Budget Shortfall,Rural-Urban Continuum Code (2013),Rural-Urban Continuum Code (2023),Census Region,Census Division,FNS Region
1316,27001,MN,"Aitkin County, Minnesota",2024,0.150,2420.0,NaN,NaN,0.11,2.0,...,0.32,imputed,3.86,24.15,1773000.0,8.0,8,Midwest,West North Central,Midwest
1317,27003,MN,"Anoka County, Minnesota",2024,0.089,32940.0,0.3,0.21,0.07,2.0,...,0.38,NaN,3.94,24.68,24662000.0,1.0,1,Midwest,West North Central,Midwest
1318,27005,MN,"Becker County, Minnesota",2024,0.135,4760.0,NaN,0.42,0.09,2.0,...,0.19,NaN,3.79,23.72,3424000.0,6.0,6,Midwest,West North Central,Midwest


## Step 3: Clean and combine

What changes here:
- **Keep 19 useful columns** and give them short names (mapping below).
  Dropped: `Ratio (1 in X)` (only in 2025 file), `Imputed Market Basket Data` (only in 2026 file), `Rural-Urban Continuum Code (2013)` (outdated), `Census Region`, `Census Division`, `FNS Region` (same value for all of MN).
- **FIPS** as a 5-digit text code (e.g. `27053` = Hennepin), which is the join key for Census/other public data.
- **County name** shortened: `"Aitkin County, Minnesota"` → `"Aitkin"`.
- **`rural`** column: USDA Rural-Urban Continuum Code 2023 ≥ 4 means non-metro (rural).
- **`source_file`** column so every number can be traced back.

⚠️ `cost_per_meal`, `weekly_dollars_needed_per_fi`, `annual_food_budget_shortfall` are **not comparable across years before 2023** (MMG methodology change).

In [4]:
COLUMNS = {
    "FIPS": "fips",
    "County, State": "county",
    "Year": "year",
    "Overall Food Insecurity Rate": "fi_rate",
    "# of Food Insecure Persons Overall": "fi_persons",
    "Child Food Insecurity Rate": "child_fi_rate",
    "# of Food Insecure Children": "fi_children",
    "Food Insecurity Rate among Black Persons (all ethnicities)": "fi_rate_black",
    "Food Insecurity Rate among Hispanic Persons (any race)": "fi_rate_hispanic",
    "Food Insecurity Rate among White, non-Hispanic Persons": "fi_rate_white_nh",
    "SNAP Threshold": "snap_threshold",
    "% FI ≤ SNAP Threshold": "pct_fi_below_snap_threshold",
    "% FI > SNAP Threshold": "pct_fi_above_snap_threshold",
    "% food insecure children in HH w/ HH incomes below 185 FPL": "pct_fi_children_below_185fpl",
    "% food insecure children in HH w/ HH incomes above 185 FPL": "pct_fi_children_above_185fpl",
    "Cost Per Meal": "cost_per_meal",
    "Weighted weekly $ needed by FI": "weekly_dollars_needed_per_fi",
    "Weighted Annual Food Budget Shortfall": "annual_food_budget_shortfall",
    "Rural-Urban Continuum Code (2023)": "rucc_2023",
}

parts = []
for df, file in [(mn_2022_2023, FILE_2025), (mn_2024, FILE_2026)]:
    part = df[list(COLUMNS)].rename(columns=COLUMNS)
    part["source_file"] = file
    parts.append(part)

mmg = pd.concat(parts, ignore_index=True)
mmg["fips"] = mmg["fips"].astype(int).astype(str).str.zfill(5)
mmg["county"] = mmg["county"].str.replace(" County, Minnesota", "", regex=False)
mmg["rural"] = mmg["rucc_2023"] >= 4
mmg = mmg.sort_values(["fips", "year"]).reset_index(drop=True)
mmg.head()

,fips,county,year,fi_rate,fi_persons,child_fi_rate,fi_children,fi_rate_black,fi_rate_hispanic,fi_rate_white_nh,...,pct_fi_below_snap_threshold,pct_fi_above_snap_threshold,pct_fi_children_below_185fpl,pct_fi_children_above_185fpl,cost_per_meal,weekly_dollars_needed_per_fi,annual_food_budget_shortfall,rucc_2023,source_file,rural
0,27001,Aitkin,2022,0.130,2060.0,0.199,510.0,NaN,NaN,0.100,...,0.754,0.246,0.76,0.24,4.37,27.07,1692000.0,8.0,MMG2025_2019-2023_Data_To_Share_v2.xlsx,True
1,27001,Aitkin,2023,0.137,2190.0,0.204,520.0,NaN,NaN,0.110,...,0.706,0.294,0.67,0.33,3.80,23.76,1579000.0,8.0,MMG2025_2019-2023_Data_To_Share_v2.xlsx,True
2,27001,Aitkin,2024,0.150,2420.0,0.220,550.0,NaN,NaN,0.110,...,0.740,0.260,0.68,0.32,3.86,24.15,1773000.0,8.0,MMG2026_2024_Data_To_Share.xlsx,True
3,27003,Anoka,2022,0.072,26300.0,0.126,10820.0,0.23,0.19,0.060,...,0.565,0.435,0.60,0.40,4.40,27.25,21743000.0,1.0,MMG2025_2019-2023_Data_To_Share_v2.xlsx,False
4,27003,Anoka,2023,0.085,31330.0,0.133,11620.0,0.27,0.22,0.068,...,0.540,0.460,0.63,0.37,3.83,23.96,22770000.0,1.0,MMG2025_2019-2023_Data_To_Share_v2.xlsx,False


## Step 4: Checks, then save

In [5]:
counts = mmg.groupby("year")["fips"].nunique()
assert (counts == 87).all(), counts
assert not mmg.duplicated(["fips", "year"]).any()
print("counties per year:", counts.to_dict())
print("rural counties:", int(mmg[mmg.year == 2024]["rural"].sum()), "of 87")
print("\nmissing values (race estimates are suppressed for small groups, don't rely on them):")
print(mmg.isna().sum()[lambda s: s > 0].to_string())

mmg.to_csv(OUT_DIR / "mmg_mn_county_2022_2024.csv", index=False)
print("\nsaved Data/processed/mmg_mn_county_2022_2024.csv:", mmg.shape)

counties per year: {2022: 87, 2023: 87, 2024: 87}
rural counties: 60 of 87

missing values (race estimates are suppressed for small groups, don't rely on them):
fi_rate_black       189
fi_rate_hispanic     97

saved Data/processed/mmg_mn_county_2022_2024.csv: (261, 21)


In [6]:
# Quick look: food insecurity rate by county and year
mmg.pivot(index="county", columns="year", values="fi_rate").sort_values(2024, ascending=False).head(10)

year,2022,2023,2024
county,,,
Mahnomen,0.157,0.165,0.167
Swift,0.113,0.125,0.157
Martin,0.117,0.125,0.154
Nobles,0.106,0.121,0.152
Clearwater,0.131,0.135,0.151
Aitkin,0.130,0.137,0.150
Beltrami,0.129,0.138,0.150
Watonwan,0.109,0.117,0.149
Chippewa,0.113,0.130,0.146


## Step 5: Statewide headline numbers (State tab)

Use these for statewide claims on slides. **Don't add up counties**: MMG builds the State tab from congressional-district
estimates, so the county sum is 3-6% lower than the State tab (compared below).

Saved to `Data/local/` (gitignored, stays on this laptop).

In [7]:
state = pd.concat([
    pd.read_excel(MMG_DIR / FILE_2025, sheet_name="State").query("State == 'MN' and Year in [2022, 2023]"),
    pd.read_excel(MMG_DIR / FILE_2026, sheet_name="State").query("State == 'MN' and Year == 2024"),
]).sort_values("Year")
state.to_csv(LOCAL_DIR / "MMG_State_MN_2022-2024_raw.csv", index=False)

compare = state.set_index("Year")[["# of Food Insecure Persons Overall"]].rename(
    columns={"# of Food Insecure Persons Overall": "state_tab"})
compare["sum_of_87_counties"] = mmg.groupby("year")["fi_persons"].sum()
compare["gap_pct"] = (1 - compare["sum_of_87_counties"] / compare["state_tab"]).mul(100).round(1)
compare

,state_tab,sum_of_87_counties,gap_pct
Year,,,
2022,537890.0,509450.0,5.3
2023,596190.0,581270.0,2.5
2024,676920.0,636610.0,6.0
